# RQ3 — Candidate Depth *k* Visualisations

In [9]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio, re

pio.templates.default = "plotly_white"

# ── Style guide constants ──────────────────────────────────────────────
COLORS = {
    "FiQA":     "#2D6A4F",
    "SciFact":  "#E07A5F",
    "ArguAna":  "#3D5A80",
    "NFCorpus": "#9B5DE5",
}
FONT = "Helvetica Neue, Segoe UI, sans-serif"

def hex_to_rgba(hex_color, alpha=0.12):
    r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)
    return f"rgba({r},{g},{b},{alpha})"

DATASETS = ["FiQA", "SciFact", "ArguAna", "NFCorpus"]
DATASET_FILE_MAP = {
    "FiQA": "fiqa", "SciFact": "scifact",
    "ArguAna": "arguana", "NFCorpus": "nfcorpus",
}
RESULTS_DIR = "results"  # ← change if your CSVs live elsewhere


In [10]:
def parse_k(name):
    """Extract k from strings like 'BM25 -> TAS-B rerank (k=100)'."""
    m = re.search(r"k=(\d+)", name)
    return int(m.group(1)) if m else None

rerank_data = {}   # dataset -> DataFrame with columns [k, RR@10, nDCG@10, AP@100, R@100]
recall_data = {}   # dataset -> DataFrame with columns [k, Recall@k]
bm25_baselines = {}  # dataset -> dict of metric baselines

for ds, slug in DATASET_FILE_MAP.items():
    # ── rerank CSV ──
    rr = pd.read_csv(f"{RESULTS_DIR}/{slug}_rerank.csv")
    bm25_row = rr[rr["name"] == "BM25"].iloc[0]
    bm25_baselines[ds] = {
        "RR@10": bm25_row["RR@10"],
        "nDCG@10": bm25_row["nDCG@10"],
        "AP@100": bm25_row["AP@100"],
        "R@100": bm25_row["R@100"],
    }
    rr_rerank = rr[rr["name"] != "BM25"].copy()
    rr_rerank["k"] = rr_rerank["name"].apply(parse_k)
    rr_rerank = rr_rerank.sort_values("k").reset_index(drop=True)
    rerank_data[ds] = rr_rerank

    # ── recall CSV ──
    rc = pd.read_csv(f"{RESULTS_DIR}/{slug}_recall.csv")
    rc["k"] = rc["name"].apply(parse_k)
    rc = rc.sort_values("k").reset_index(drop=True)
    recall_data[ds] = rc

print("Loaded datasets:", list(rerank_data.keys()))
for ds in DATASETS:
    ks = sorted(rerank_data[ds]["k"].unique())
    print(f"  {ds}: k = {ks[0]}..{ks[-1]} ({len(ks)} points), BM25 nDCG@10 = {bm25_baselines[ds]['nDCG@10']:.4f}")


Loaded datasets: ['FiQA', 'SciFact', 'ArguAna', 'NFCorpus']
  FiQA: k = 50..800 (31 points), BM25 nDCG@10 = 0.2526
  SciFact: k = 50..800 (31 points), BM25 nDCG@10 = 0.6722
  ArguAna: k = 50..550 (21 points), BM25 nDCG@10 = 0.3424
  NFCorpus: k = 50..800 (31 points), BM25 nDCG@10 = 0.3222


In [11]:
def style_layout(fig, title, subtitle=None, width=820, height=460):
    """Apply style-guide layout to a figure."""
    full_title = f"<b>{title}</b>"
    if subtitle:
        full_title += f"<br><span style='font-size:12px;color:#666'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=full_title, font=dict(size=18)),
        font=dict(family=FONT),
        width=width,
        height=height,
        margin=dict(t=80, b=60),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
        ),
    )
    return fig


## 1 — Raw Metrics Grid

In [12]:
METRICS = ["RR@10", "nDCG@10", "AP@100", "R@100"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"<b>{m}</b>" for m in METRICS],
    horizontal_spacing=0.10, vertical_spacing=0.14,
)

for i, metric in enumerate(METRICS):
    row, col = divmod(i, 2)
    row += 1; col += 1
    for ds in DATASETS:
        df = rerank_data[ds]
        fig.add_trace(go.Scatter(
            x=df["k"], y=df[metric],
            name=ds, legendgroup=ds,
            showlegend=(i == 0),
            line=dict(color=COLORS[ds], width=2.5),
            hovertemplate=(
                f"<b>{ds}</b><br>k=%{{x}}<br>{metric}=%{{y:.4f}}<extra></extra>"
            ),
        ), row=row, col=col)

        # BM25 baseline
        bl = bm25_baselines[ds][metric]
        fig.add_hline(
            y=bl, row=row, col=col,
            line=dict(color=COLORS[ds], width=1, dash="dot"),
            annotation=None,
        )

    fig.update_yaxes(title_text=metric, row=row, col=col)
    fig.update_xaxes(title_text="k", row=row, col=col)

style_layout(fig,
    "Raw Reranking Metrics across Candidate Depth k",
    width=900, height=600)
fig.show()


## 2 — Delta over BM25 Baseline

In [13]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"<b>Δ {m}</b>" for m in METRICS],
    horizontal_spacing=0.10, vertical_spacing=0.14,
)

for i, metric in enumerate(METRICS):
    row, col = divmod(i, 2)
    row += 1; col += 1
    for ds in DATASETS:
        df = rerank_data[ds]
        bl = bm25_baselines[ds][metric]
        delta = df[metric] - bl

        fig.add_trace(go.Scatter(
            x=df["k"], y=delta,
            name=ds, legendgroup=ds,
            showlegend=(i == 0),
            line=dict(color=COLORS[ds], width=2.5),
            fill="tozeroy",
            fillcolor=hex_to_rgba(COLORS[ds], 0.12),
            hovertemplate=(
                f"<b>{ds}</b><br>k=%{{x}}<br>Δ{metric}=%{{y:+.4f}}<extra></extra>"
            ),
        ), row=row, col=col)

    # zero reference line
    fig.add_hline(y=0, row=row, col=col,
                  line=dict(color="#999", width=1, dash="dot"))
    fig.update_yaxes(title_text=f"Δ {metric}", row=row, col=col)
    fig.update_xaxes(title_text="k", row=row, col=col)

style_layout(fig,
    "Change in Metrics Relative to BM25 Baseline",
    width=900, height=600)
fig.show()


## 3 — Recall@k per Dataset

In [14]:
fig = go.Figure()

for ds in DATASETS:
    df = recall_data[ds]
    fig.add_trace(go.Scatter(
        x=df["k"], y=df["Recall@k"],
        name=ds,
        line=dict(color=COLORS[ds], width=2.5),
        mode="lines+markers",
        marker=dict(
            color=COLORS[ds], size=5,
            line=dict(color="white", width=1),
        ),
        hovertemplate=(
            f"<b>{ds}</b><br>k=%{{x}}<br>Recall@k=%{{y:.4f}}<extra></extra>"
        ),
    ))

fig.update_xaxes(title_text="Candidate depth k")
fig.update_yaxes(title_text="Recall@k")

style_layout(fig,
    "Recall@k — Proportion of Relevant Documents in Top-k Candidates")
fig.show()


## 4 — Normalised % Change in nDCG@10

In [15]:
fig = go.Figure()

for ds in DATASETS:
    df = rerank_data[ds]
    bl = bm25_baselines[ds]["nDCG@10"]
    pct_change = ((df["nDCG@10"] - bl) / bl) * 100

    fig.add_trace(go.Scatter(
        x=df["k"], y=pct_change,
        name=ds,
        line=dict(color=COLORS[ds], width=2.5),
        fill="tozeroy",
        fillcolor=hex_to_rgba(COLORS[ds], 0.12),
        hovertemplate=(
            f"<b>{ds}</b><br>k=%{{x}}<br>Δ nDCG@10 = %{{y:+.1f}}%<extra></extra>"
        ),
    ))

    # Annotate peak/trough
    best_idx = pct_change.abs().idxmax() if (pct_change > 0).any() else pct_change.idxmin()
    if (pct_change > 0).any():
        best_idx = pct_change.idxmax()
    else:
        best_idx = pct_change.idxmin()

    fig.add_trace(go.Scatter(
        x=[df["k"].iloc[best_idx]],
        y=[pct_change.iloc[best_idx]],
        mode="markers+text",
        marker=dict(color=COLORS[ds], size=8,
                    line=dict(color="white", width=1.5)),
        text=[f"{pct_change.iloc[best_idx]:+.1f}%"],
        textposition="top center",
        textfont=dict(size=10, color=COLORS[ds]),
        showlegend=False,
        hoverinfo="skip",
    ))

# zero reference
fig.add_hline(y=0, line=dict(color="#999", width=1, dash="dot"))
fig.update_xaxes(title_text="Candidate depth k")
fig.update_yaxes(title_text="% change in nDCG@10 vs BM25")

style_layout(fig,
    "Normalised Percentage Change in nDCG@10 Relative to BM25")
fig.show()
